In [1]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import pickle
from time import time

In [2]:
DEFAULT_DATA = '/home/hep/lprate/projects/MuonsAndMatter/data/muons/full_sample_w.h5'

In [ ]:
data = {}
with h5py.File(DEFAULT_DATA, 'r') as f:
    for group_name in f.keys():
        print(f"Group: {group_name}")
        group = f[group_name]
        weight = group.attrs['weight']
        if float(weight) < 0.0001: continue
        px = group['px'][:]
        py = group['py'][:]
        pz = group['pz'][:]
        x_ = group['x'][:]
        y_ = group['y'][:]
        z_ = group['z'][:]
        pdg = group['pdg'][:]
        data[weight] = np.stack([px, py, pz, x_, y_, z_, pdg], axis=1)
        print(f"Weight: {weight}, Shape: {data[weight].shape}")

del px, py, pz, x_, y_, z_, pdg
pz_real = np.concatenate([data[weight][:, 2] for weight in data.keys()])
pt_real = np.sqrt(np.concatenate([data[weight][:, 0]**2 + data[weight][:, 1]**2 for weight in data.keys()]))
weights = np.concatenate([np.full(data[w].shape[0], float(w), dtype=float) for w in data.keys()])


Group: 0
Group: 1
Weight: 0.09369999915361404, Shape: (3311, 7)
Group: 2
Weight: 1.0, Shape: (20993, 7)
Group: 3
Weight: 3.2616000175476074, Shape: (691113, 7)
Group: 4
Weight: 7.6875, Shape: (480474935, 7)
Group: 5
Weight: 9.369999885559082, Shape: (168445, 7)
Group: 6
Weight: 326.1600036621094, Shape: (11094133, 7)
Group: 7
Weight: 768.75, Shape: (13921031, 7)


In [ ]:
with open('outputs/gmm_model.pkl', 'rb') as f:
    models = pickle.load(f)
models.pop(0.0, None)  # Remove the model with weight 0.0 if it exists

In [ ]:
n_samples = 300e6
alpha = n_samples / sum([model['weight'] for model in models.values()])
gen_pz = []
gen_pt = []
for weight, model in models.items():
    real_weight = round(model['weight'] * alpha)
    if real_weight<1: continue
    gmm = model['gmm']
    mean,std = model['mean'], model['std']
    t1 = time()
    sample, _ = gmm.sample(real_weight)
    sample = sample * std + mean
    print(f"Weight: {weight}")
    print(f"Sampling {real_weight} samples took {time()-t1:.2f} seconds")
    gen_pz.append(sample[:, 0])
    gen_pt.append(sample[:, 1])

del sample, mean, std
gen_pz = np.concatenate(gen_pz)
gen_pt = np.concatenate(gen_pt)
print(f"Generated pz shape: {gen_pz.shape}, pt shape: {gen_pt.shape}")

In [ ]:
from matplotlib.colors import LogNorm

def compare_1d_histograms(real_sample, gen_sample, bins=120, range_vals=None, xlabel="Value", real_w=None):
    """Overlays 1D distributions of real and generated samples."""
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # Auto-detect range if none is provided
    if range_vals is None:
        range_vals = (min(np.min(real_sample), np.min(gen_sample)), 
                      max(np.max(real_sample), np.max(gen_sample)))
        
    b = np.linspace(range_vals[0], range_vals[1], bins)
    
    # Plot marginals
    ax.hist(real_sample, bins=b, weights=real_w, density=True, 
            histtype='step', color='k', label='Real')
    ax.hist(gen_sample, bins=b, density=True, 
            histtype='step', color='C1', label='Generated')
    
    ax.set_xlabel(xlabel, fontsize=14)
    ax.set_ylabel('Density', fontsize=14)
    ax.legend(fontsize=11)
    
    fig.tight_layout()
    return fig, ax


def compare_2d_histograms(real_pz, real_pt, gen_pz, gen_pt, real_w=None):
    """Creates side-by-side 2D Pz-Pt density histograms with a shared color scale."""
    pz_bins = np.linspace(0, 400, 201)
    pt_bins = np.linspace(0, 13, 201)
    
    # Calculate 2D densities
    Hr, _, _ = np.histogram2d(real_pz, real_pt, bins=[pz_bins, pt_bins], 
                              weights=real_w, density=True)
    Hg, _, _ = np.histogram2d(gen_pz, gen_pt, bins=[pz_bins, pt_bins], 
                              density=True)
    
    # Shared LogNorm for accurate visual comparison
    pos_vals = np.concatenate([Hr[Hr > 0].ravel(), Hg[Hg > 0].ravel()])
    norm = LogNorm(vmin=pos_vals.min(), vmax=max(Hr.max(), Hg.max()))
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    
    # Draw meshes
    for ax, H, title in zip(axes, [Hr, Hg], ['Real', 'Generated']):
        im = ax.pcolormesh(pz_bins, pt_bins, H.T, cmap='viridis', norm=norm)
        ax.set_xlim(0, 400)
        ax.set_ylim(0, 13)
        ax.set_xlabel('$P_z$ [GeV]', fontsize=14)
        ax.set_ylabel('$P_t$ [GeV]', fontsize=14)
        ax.set_title(title, fontsize=14)
        
    # Shared colorbar
    fig.colorbar(im, ax=axes.tolist(), label='Density')
    
    return fig, axes

In [ ]:
compare_2d_histograms(pz_real, pt_real, gen_pz, gen_pt, real_w=weights)

# Weight 0.09

In [ ]:
i = 0
w = list(data.keys())[i]
model = models[w]
w

In [ ]:
n_samples = 50e6
gmm = model['gmm']
mean,std = model['mean'], model['std']
sample = gmm.sample(n_samples)[0]
sample = sample * std + mean
gen_pz = sample[:, 0]
gen_pt = sample[:, 1]

In [ ]:
pz_real = np.concatenate([data[w][:, 2]])
pt_real = np.sqrt(np.concatenate([data[w][:, 0]**2 + data[w][:, 1]**2]))

In [ ]:
compare_2d_histograms(pz_real, pt_real, gen_pz, gen_pt)
print(f"Number of real samples: {pz_real.shape[0]}, Number of generated samples: {gen_pz.shape[0]}")

# Weight 1.0

In [ ]:
i = 1
w = list(data.keys())[i]
model = models[w]
w

In [ ]:
n_samples = 50e6
gmm = model['gmm']
mean,std = model['mean'], model['std']
sample = gmm.sample(n_samples)[0]
sample = sample * std + mean
gen_pz = sample[:, 0]
gen_pt = sample[:, 1]

In [ ]:
pz_real = np.concatenate([data[w][:, 2]])
pt_real = np.sqrt(np.concatenate([data[w][:, 0]**2 + data[w][:, 1]**2]))

In [ ]:
compare_2d_histograms(pz_real, pt_real, gen_pz, gen_pt)
print(f"Number of real samples: {pz_real.shape[0]}, Number of generated samples: {gen_pz.shape[0]}")

# Weight 3.26

In [ ]:
i = 2
w = list(data.keys())[i]
model = models[w]
w

In [ ]:
n_samples = 50e6
gmm = model['gmm']
mean,std = model['mean'], model['std']
sample = gmm.sample(n_samples)[0]
sample = sample * std + mean
gen_pz = sample[:, 0]
gen_pt = sample[:, 1]

In [ ]:
pz_real = np.concatenate([data[w][:, 2]])
pt_real = np.sqrt(np.concatenate([data[w][:, 0]**2 + data[w][:, 1]**2]))

In [ ]:
compare_2d_histograms(pz_real, pt_real, gen_pz, gen_pt)
print(f"Number of real samples: {pz_real.shape[0]}, Number of generated samples: {gen_pz.shape[0]}")

# Weight 7.69

In [ ]:
i = 3
w = list(data.keys())[i]
model = models[w]
w

In [ ]:
n_samples = 500e6
gmm = model['gmm']
mean,std = model['mean'], model['std']
sample = gmm.sample(n_samples)[0]
sample = sample * std + mean
gen_pz = sample[:, 0]
gen_pt = sample[:, 1]

In [ ]:
pz_real = np.concatenate([data[w][:, 2]])
pt_real = np.sqrt(np.concatenate([data[w][:, 0]**2 + data[w][:, 1]**2]))

In [ ]:
compare_2d_histograms(pz_real, pt_real, gen_pz, gen_pt)
print(f"Number of real samples: {pz_real.shape[0]}, Number of generated samples: {gen_pz.shape[0]}")

# Weight 9.37

In [ ]:
i = 4
w = list(data.keys())[i]
model = models[w]
w

In [ ]:
n_samples = 50e6
gmm = model['gmm']
mean,std = model['mean'], model['std']
sample = gmm.sample(n_samples)[0]
sample = sample * std + mean
gen_pz = sample[:, 0]
gen_pt = sample[:, 1]

In [ ]:
pz_real = np.concatenate([data[w][:, 2]])
pt_real = np.sqrt(np.concatenate([data[w][:, 0]**2 + data[w][:, 1]**2]))

In [ ]:
compare_2d_histograms(pz_real, pt_real, gen_pz, gen_pt)
print(f"Number of real samples: {pz_real.shape[0]}, Number of generated samples: {gen_pz.shape[0]}")

# Weight 326.16

In [ ]:
i = 5
w = list(data.keys())[i]
model = models[w]
w

In [ ]:
n_samples = 50e6
gmm = model['gmm']
mean,std = model['mean'], model['std']
sample = gmm.sample(n_samples)[0]
sample = sample * std + mean
gen_pz = sample[:, 0]
gen_pt = sample[:, 1]

In [ ]:
pz_real = np.concatenate([data[w][:, 2]])
pt_real = np.sqrt(np.concatenate([data[w][:, 0]**2 + data[w][:, 1]**2]))

In [ ]:
compare_2d_histograms(pz_real, pt_real, gen_pz, gen_pt)
print(f"Number of real samples: {pz_real.shape[0]}, Number of generated samples: {gen_pz.shape[0]}")

# Weight 768.75

In [ ]:
i = 6
w = list(data.keys())[i]
model = models[w]
w

In [ ]:
n_samples = 50e6
gmm = model['gmm']
mean,std = model['mean'], model['std']
sample = gmm.sample(n_samples)[0]
sample = sample * std + mean
gen_pz = sample[:, 0]
gen_pt = sample[:, 1]

In [ ]:
pz_real = np.concatenate([data[w][:, 2]])
pt_real = np.sqrt(np.concatenate([data[w][:, 0]**2 + data[w][:, 1]**2]))

In [ ]:
compare_2d_histograms(pz_real, pt_real, gen_pz, gen_pt)
print(f"Number of real samples: {pz_real.shape[0]}, Number of generated samples: {gen_pz.shape[0]}")